# Linear RRF Weight Model With LLM Judge

This notebook extends `linear_rrf_weight_model.ipynb` with the LLM judge setting.

The goal is to learn how much each query variant should contribute to weighted Reciprocal Rank Fusion (RRF), then test whether an additional LLM judge refinement on the held-out test set improves recall.

Let:

- $q$ be the original process query.
- $D = \{d_1, \dots, d_N\}$ be the regulatory corpus.
- $V = \{v_0, v_1, v_2, v_3, v_4\}$ be the query variants.
- $v_0$ is the original query, and $v_1, \dots, v_4$ are the four LLM-generated query variants.
- $r_v(q, d)$ is the BM25 rank of document $d$ for variant $v$.
- $G(q) \subseteq D$ is the gold-standard relevant document set for query $q$.

Workflow:

- Load recorded query variants from `output_with_agents_<use_case>.csv`.
- Use the first LLM judge pass to create judge-refined variants when they are missing.
- Split each use case and process level into train and test parts.
- Learn linear RRF weights on the train split using the judge-refined variants.
- Apply the learned weights on the test split.
- Optionally run a second LLM judge pass on the test queries, then evaluate the improved test queries.

The token-spending steps are controlled by flags, so opening the notebook does not call an LLM automatically.

## Setup

This cell imports the dependencies, finds the project root, and imports the existing retrieval and LLM judge helpers from `main.py`.

In [21]:
from pathlib import Path
import importlib
import os
import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

project_root = Path.cwd()
while project_root.name != "Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval" and project_root.parent != project_root:
    project_root = project_root.parent

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import main
importlib.reload(main)

from main import (
    JudgeContext,
    QueryVariants,
    clean_text,
    cross_encoder_rows_to_df,
    llm_judge_refine_queries,
)
from retrieval.query_merging import RankedList, ReciprocalRankFusion
from retrieval.retrieval_bm25 import Query, build_bm25_index, load_corpus
from retrieval.sota_retrieval import SotaRetriever

print(f"Working directory: {Path.cwd()}")

Working directory: /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval


## Configuration

`TRAIN_USE_CASES` controls where the linear weights are learned. `APPLY_WEIGHT_USE_CASES` controls additional use cases where those learned weights are applied without learning use-case-specific weights. Here, weights are learned on UC1/UC2 and then applied to UC3.

`RUN_LLM_JUDGE_FOR_MISSING` creates missing first-pass judge columns in the records file. `RUN_SECOND_TEST_JUDGE` runs the extra judge pass on held-out test queries after the linear weights have been learned.

The experiment uses five retrieval variants:

$$
V = \{\text{baseline}, \text{legal}, \text{compliance}, \text{contract}, \text{risk}\}
$$

The learned weight vector has one component per variant:

$$
\mathbf{w} = (w_0, w_1, w_2, w_3, w_4)
$$

The weights are constrained after learning so that:

$$
w_i \ge 0 \quad \text{and} \quad \sum_{i=0}^{4} w_i = 1
$$

This makes the final score interpretable as a weighted combination of RRF evidence from the query variants.

Both LLM flags are controlled manually to avoid accidental token usage.

In [22]:
TRAIN_USE_CASES = ["uc1", "uc2"]
APPLY_WEIGHT_USE_CASES = ["uc3"]
LEVELS = ["process", "subprocess", "task"]
RRF_K = 60
RANDOM_STATE = 42
JUDGE_PROVIDER = "gemini"
RUN_LLM_JUDGE_FOR_MISSING = True
RUN_SECOND_TEST_JUDGE = True

BASE_VARIANTS = [
    ("baseline", "query"),
    ("legal_terminology_rewrite", "legal_terminology_rewrite"),
    ("regulatory_compliance_query", "regulatory_compliance_query"),
    ("contract_clause_query", "contract_clause_query"),
    ("risk_scenario_query", "risk_scenario_query"),
]

JUDGE_VARIANTS = [
    ("baseline", "query"),
    ("judge_legal_terminology_rewrite", "judge_legal_terminology_rewrite"),
    ("judge_regulatory_compliance_query", "judge_regulatory_compliance_query"),
    ("judge_contract_clause_query", "judge_contract_clause_query"),
    ("judge_risk_scenario_query", "judge_risk_scenario_query"),
]

SECOND_JUDGE_VARIANTS = [
    ("baseline", "query"),
    ("test_judge_legal_terminology_rewrite", "test_judge_legal_terminology_rewrite"),
    ("test_judge_regulatory_compliance_query", "test_judge_regulatory_compliance_query"),
    ("test_judge_contract_clause_query", "test_judge_contract_clause_query"),
    ("test_judge_risk_scenario_query", "test_judge_risk_scenario_query"),
]

LEVEL_TO_GS_SUFFIX = {
    "process": "process_level",
    "subprocess": "subprocess_level",
    "task": "event_level",
}

LEVEL_TO_TOP_K = {
    "process": 100,
    "subprocess": 30,
    "task": 15,
}

FIRST_PASS_EQUAL_WEIGHTS = np.ones(len(BASE_VARIANTS)) / len(BASE_VARIANTS)

## Data And Ranking Helpers

These helpers load the corpus, gold standard, and recorded variants. The judge enrichment function uses the same judge prompt and CE context as the main pipeline.

For each query $q$ and variant $v_i$, BM25 returns a ranked list:

$$
R_i(q) = (d_{i,1}, d_{i,2}, \dots, d_{i,k})
$$

where $d_{i,j}$ is the document at rank $j$ for variant $v_i$. If document $d$ appears in this list, its rank is:

$$
r_i(q,d) = j
$$

If $d$ does not appear in the top-$k$ list for variant $v_i$, it contributes no evidence for that variant.

The first LLM judge pass observes retrieval evidence from the current pipeline and rewrites the four non-baseline variants. Conceptually:

$$
(v_1^{(1)}, \dots, v_4^{(1)}) = J\left(q, v_1^{(0)}, \dots, v_4^{(0)}, \operatorname{Top}_{15}^{BM25}(q), \operatorname{Top}_{15}^{CE}(q)\right)
$$

Here $J(\cdot)$ is the LLM judge, $v_i^{(0)}$ are the original variants, and $v_i^{(1)}$ are the first-pass judged variants used for linear weight learning.

In [23]:
def corpus_path_for(use_case):
    return Path(f"regulatory_relevance4process/SOTA_NLP_LIR/input_ranking/{use_case}/Input_corpus_{use_case}.xlsx")


def gold_path_for(use_case, level):
    suffix = LEVEL_TO_GS_SUFFIX[level]
    return Path(f"regulatory_relevance4process/SOTA_NLP_LIR/output_ranking_input_eval/{use_case}/gold_standard/gs_{use_case}_{suffix}.xlsx")


def recorded_path_for(use_case):
    return Path(f"output_with_agents_{use_case}.csv")


def load_records(use_case):
    path = recorded_path_for(use_case)
    if not path.exists():
        raise FileNotFoundError(f"Missing recorded query variants: {path}")

    df = pd.read_csv(path)
    df["level"] = df["level"].astype(str).str.lower()
    df["query_clean"] = df["query"].apply(clean_text)
    return df


def save_records(use_case, records_df):
    path = recorded_path_for(use_case)
    records_df.drop(columns=["query_clean"], errors="ignore").to_csv(path, index=False)
    print(f"Saved judge-enriched records to {path}")


def split_queries(queries):
    queries = list(queries)
    if len(queries) == 1:
        return queries, queries, "single_query_reused"

    rng = np.random.default_rng(RANDOM_STATE)
    indices = np.arange(len(queries))
    rng.shuffle(indices)
    split_at = max(1, len(indices) // 2)
    train_indices = set(indices[:split_at])
    train_queries = [query for idx, query in enumerate(queries) if idx in train_indices]
    test_queries = [query for idx, query in enumerate(queries) if idx not in train_indices]
    return train_queries, test_queries, "random_half_split"


def variant_text(row, column):
    value = row[column] if column in row.index else ""
    if value is None:
        return clean_text(row["query"])
    if isinstance(value, float) and pd.isna(value):
        return clean_text(row["query"])
    if str(value).strip() == "":
        return clean_text(row["query"])
    return clean_text(value)


def query_variants_from_row(row, variant_columns):
    variant_lookup = {name: column for name, column in variant_columns}
    return QueryVariants(
        legal_terminology_rewrite=variant_text(row, variant_lookup.get("legal_terminology_rewrite", "judge_legal_terminology_rewrite")),
        regulatory_compliance_query=variant_text(row, variant_lookup.get("regulatory_compliance_query", "judge_regulatory_compliance_query")),
        contract_clause_query=variant_text(row, variant_lookup.get("contract_clause_query", "judge_contract_clause_query")),
        risk_scenario_query=variant_text(row, variant_lookup.get("risk_scenario_query", "judge_risk_scenario_query")),
    )


def rank_variant_lists(bm25_index, row, top_k, variant_columns):
    ranked_lists = {}
    for variant_name, column in variant_columns:
        ranked_lists[variant_name] = bm25_index.rank(Query(text=variant_text(row, column)), top_k=top_k)
    return ranked_lists


def rrf_feature_rows(ranked_lists, variant_columns):
    doc_features = {}
    for variant_index, (variant_name, _) in enumerate(variant_columns):
        for result in ranked_lists[variant_name]:
            features = doc_features.setdefault(result.document.doc_id, np.zeros(len(variant_columns), dtype=float))
            features[variant_index] = 1.0 / (RRF_K + result.rank)
    return doc_features

In [24]:
def make_query_variants(row, legal_col, regulatory_col, contract_col, risk_col):
    return QueryVariants(
        legal_terminology_rewrite=variant_text(row, legal_col),
        regulatory_compliance_query=variant_text(row, regulatory_col),
        contract_clause_query=variant_text(row, contract_col),
        risk_scenario_query=variant_text(row, risk_col),
    )


def weighted_ce_rows_for_context(level, row, bm25_index, documents, sota_retriever, top_k, variant_columns, weights):
    ranked_lists = []
    for variant_name, column in variant_columns:
        ranked_lists.append(
            RankedList(
                name=variant_name,
                results=bm25_index.rank(Query(text=variant_text(row, column)), top_k=top_k),
            )
        )

    rrf = ReciprocalRankFusion()
    merged = rrf.fuse(ranked_lists, top_k=top_k, weights=weights)
    return cross_encoder_rows_to_df(
        level_name=level,
        query_text=clean_text(row["query"]),
        results=merged,
        documents=documents,
        cross_encoder=sota_retriever.cross_encoder,
        top_k=top_k,
        method="linear_rrf_context_ce",
        query_variant="linear_weighted_context",
    )


def run_judge_for_row(level, row, bm25_index, documents, sota_retriever, top_k, variant_columns, weights, current_variants):
    baseline_results = bm25_index.rank(Query(text=clean_text(row["query"])), top_k=top_k)
    ce_rows = weighted_ce_rows_for_context(
        level=level,
        row=row,
        bm25_index=bm25_index,
        documents=documents,
        sota_retriever=sota_retriever,
        top_k=top_k,
        variant_columns=variant_columns,
        weights=weights,
    )
    return llm_judge_refine_queries(
        JudgeContext(
            original_query=clean_text(row["query"]),
            variants=current_variants,
            bm25_results=baseline_results,
            ce_rows=ce_rows,
        ),
        documents=documents,
        judge_provider=JUDGE_PROVIDER,
    )


def ensure_first_pass_judge_records(use_case, records_df):
    required_columns = [column for _, column in JUDGE_VARIANTS if column != "query"]
    missing_columns = [column for column in required_columns if column not in records_df.columns]
    if missing_columns:
        for column in missing_columns:
            records_df[column] = ""

    needs_judge = records_df[required_columns].isna().any(axis=1) | (records_df[required_columns].astype(str).apply(lambda col: col.str.strip()).eq("").any(axis=1))
    if not needs_judge.any():
        print(f"{use_case}: first-pass judge columns already available.")
        return records_df

    if not RUN_LLM_JUDGE_FOR_MISSING:
        raise ValueError(
            f"{use_case} has missing first-pass judge columns. Set RUN_LLM_JUDGE_FOR_MISSING = True to create them."
        )

    documents = load_corpus(str(corpus_path_for(use_case)))
    bm25_index = build_bm25_index(str(corpus_path_for(use_case)))
    sota_retriever = SotaRetriever([document.text for document in documents])

    for row_index, row in records_df[needs_judge].iterrows():
        level = row["level"]
        top_k = LEVEL_TO_TOP_K[level]
        print(f"[{use_case} / {level}] first-pass judge for row {row_index}")
        judged = run_judge_for_row(
            level=level,
            row=row,
            bm25_index=bm25_index,
            documents=documents,
            sota_retriever=sota_retriever,
            top_k=top_k,
            variant_columns=BASE_VARIANTS,
            weights=np.array(main.WEIGHTED_RRF_WEIGHTS, dtype=float),
            current_variants=make_query_variants(
                row,
                "legal_terminology_rewrite",
                "regulatory_compliance_query",
                "contract_clause_query",
                "risk_scenario_query",
            ),
        )
        records_df.loc[row_index, "judge_legal_terminology_rewrite"] = judged.legal_terminology_rewrite
        records_df.loc[row_index, "judge_regulatory_compliance_query"] = judged.regulatory_compliance_query
        records_df.loc[row_index, "judge_contract_clause_query"] = judged.contract_clause_query
        records_df.loc[row_index, "judge_risk_scenario_query"] = judged.risk_scenario_query

    save_records(use_case, records_df)
    return records_df

## Linear Weight Learning

The model uses the judge-refined variants as features. Each feature is the reciprocal-rank value for one variant and one candidate document. Logistic regression learns weights, which are clipped to nonnegative values, normalized, and reused as weighted RRF coefficients.

For each query-document pair $(q,d)$, the feature for variant $v_i$ is:

$$
x_i(q,d) =
\begin{cases}
\frac{1}{K + r_i(q,d)} & \text{if } d \in R_i(q) \\
0 & \text{otherwise}
\end{cases}
$$

where:

- $K$ is the RRF smoothing constant, set by `RRF_K`.
- $r_i(q,d)$ is the rank of document $d$ for variant $v_i$.
- Lower ranks produce larger feature values.

The feature vector is:

$$
\mathbf{x}(q,d) = \left[x_0(q,d), x_1(q,d), x_2(q,d), x_3(q,d), x_4(q,d)\right]
$$

The training label is binary:

$$
y(q,d) =
\begin{cases}
1 & \text{if } d \in G(q) \\
0 & \text{if } d \notin G(q)
\end{cases}
$$

A logistic regression model estimates relevance probability:

$$
P(y=1 \mid q,d) = \sigma\left(\beta_0 + \boldsymbol{\beta}^{\top}\mathbf{x}(q,d)\right)
$$

with the sigmoid function:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

The model is trained by minimizing the regularized negative log-likelihood:

$$
\mathcal{L}(\boldsymbol{\beta}) = -\sum_{(q,d)} \left[y(q,d)\log p(q,d) + (1-y(q,d))\log(1-p(q,d))\right] + \lambda \lVert \boldsymbol{\beta} \rVert_2^2
$$

where:

$$
p(q,d) = P(y=1 \mid q,d)
$$

After training, negative coefficients are clipped because a variant should not contribute negative retrieval evidence in weighted RRF:

$$
\tilde{\beta}_i = \max(\beta_i, 0)
$$

The final RRF weights are normalized:

$$
w_i = \frac{\tilde{\beta}_i}{\sum_{j=0}^{4}\tilde{\beta}_j}
$$

If all clipped coefficients are zero, the notebook falls back to equal weights:

$$
w_i = \frac{1}{|V|}
$$

In [25]:
def build_gold_lookup(use_case, level, documents):
    gold_df = pd.read_excel(gold_path_for(use_case, level))
    gold_df["query_clean"] = gold_df["query"].apply(clean_text)
    doc_id_by_text = {clean_text(document.text): document.doc_id for document in documents}

    gold_lookup = {}
    for query, group in gold_df.groupby("query_clean"):
        gold_doc_ids = {
            doc_id_by_text[clean_text(rel_text)]
            for rel_text in group["rel_text"].tolist()
            if clean_text(rel_text) in doc_id_by_text
        }
        gold_lookup[query] = gold_doc_ids
    return gold_lookup


def build_examples(records_df, bm25_index, gold_lookup, query_subset, top_k, variant_columns):
    examples = []
    labels = []

    for query in query_subset:
        row = records_df[records_df["query_clean"] == query].iloc[0]
        ranked_lists = rank_variant_lists(bm25_index, row, top_k=top_k, variant_columns=variant_columns)
        doc_features = rrf_feature_rows(ranked_lists, variant_columns=variant_columns)
        positives = gold_lookup.get(query, set())

        for doc_id, features in doc_features.items():
            examples.append(features)
            labels.append(1 if doc_id in positives else 0)

    return np.array(examples), np.array(labels)


def learn_weights(x_train, y_train, variant_columns):
    if len(x_train) == 0 or len(np.unique(y_train)) < 2:
        return np.ones(len(variant_columns)) / len(variant_columns), "equal_weights_fallback"

    model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
    model.fit(x_train, y_train)
    raw_weights = np.maximum(model.coef_[0], 0.0)

    if raw_weights.sum() == 0:
        return np.ones(len(variant_columns)) / len(variant_columns), "equal_weights_fallback"

    return raw_weights / raw_weights.sum(), "linear_logistic_coefficients"


def rank_with_weights(records_df, bm25_index, query_subset, top_k, weights, variant_columns):
    predictions = {}
    ranked_outputs = {}
    for query in query_subset:
        row = records_df[records_df["query_clean"] == query].iloc[0]
        ranked_lists = rank_variant_lists(bm25_index, row, top_k=top_k, variant_columns=variant_columns)
        doc_features = rrf_feature_rows(ranked_lists, variant_columns=variant_columns)
        scored_docs = [(doc_id, float(np.dot(features, weights))) for doc_id, features in doc_features.items()]
        scored_docs.sort(key=lambda item: item[1], reverse=True)
        ranked_outputs[query] = scored_docs[:top_k]
        predictions[query] = {doc_id for doc_id, _ in scored_docs[:top_k]}
    return predictions, ranked_outputs


def evaluate_predictions(predictions, gold_lookup):
    tp = fp = fn = 0
    for query, predicted in predictions.items():
        gold = gold_lookup.get(query, set())
        tp += len(predicted & gold)
        fp += len(predicted - gold)
        fn += len(gold - predicted)

    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else np.nan
    return {
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

## Second Judge Pass On The Test Set

This step is optional and only runs when `RUN_SECOND_TEST_JUDGE = True`. It uses the learned linear weights to build a test-set retrieval context, asks the LLM judge to refine the query variants again, and then reranks with the same learned weights.

First, the learned weights score each candidate document using weighted RRF:

$$
s_{\mathbf{w}}(q,d) = \sum_{i=0}^{4} w_i x_i(q,d)
$$

The top documents are then selected by descending score:

$$
\operatorname{rank}_{\mathbf{w}}(q) = \operatorname{argsort}_{d \in D}\left(s_{\mathbf{w}}(q,d)\right)
$$

For the second test-set judge pass, the LLM receives the original query, the first-pass judged variants, the top BM25 results, and the top weighted retrieval context. It produces another set of query variants:

$$
(v_1^{(2)}, \dots, v_4^{(2)}) = J\left(q, v_1^{(1)}, \dots, v_4^{(1)}, \operatorname{Top}_{15}^{BM25}(q), \operatorname{Top}_{15}^{\mathbf{w}}(q)\right)
$$

The same learned weights are then applied again, but with the second-pass variants:

$$
s_{\mathbf{w}}^{(2)}(q,d) = \sum_{i=0}^{4} w_i x_i^{(2)}(q,d)
$$

This isolates the effect of the second judge pass: the weights stay fixed, while only the test queries are refined again.

In [26]:
def add_second_pass_judge_columns(records_df):
    for _, column in SECOND_JUDGE_VARIANTS:
        if column != "query" and column not in records_df.columns:
            records_df[column] = ""
    return records_df


def apply_second_test_judge(use_case, level, records_df, test_queries, weights, documents, bm25_index, sota_retriever, top_k):
    records_df = add_second_pass_judge_columns(records_df.copy())
    if not RUN_SECOND_TEST_JUDGE:
        return records_df, "second_test_judge_disabled"

    for query in test_queries:
        row_index = records_df[records_df["query_clean"] == query].index[0]
        row = records_df.loc[row_index]
        print(f"[{use_case} / {level}] second-pass test judge for query: {query[:80]}")
        judged = run_judge_for_row(
            level=level,
            row=row,
            bm25_index=bm25_index,
            documents=documents,
            sota_retriever=sota_retriever,
            top_k=top_k,
            variant_columns=JUDGE_VARIANTS,
            weights=weights,
            current_variants=make_query_variants(
                row,
                "judge_legal_terminology_rewrite",
                "judge_regulatory_compliance_query",
                "judge_contract_clause_query",
                "judge_risk_scenario_query",
            ),
        )
        records_df.loc[row_index, "test_judge_legal_terminology_rewrite"] = judged.legal_terminology_rewrite
        records_df.loc[row_index, "test_judge_regulatory_compliance_query"] = judged.regulatory_compliance_query
        records_df.loc[row_index, "test_judge_contract_clause_query"] = judged.contract_clause_query
        records_df.loc[row_index, "test_judge_risk_scenario_query"] = judged.risk_scenario_query

    return records_df, "second_test_judge_enabled"


def second_pass_columns_available(records_df, test_queries):
    required_columns = [column for _, column in SECOND_JUDGE_VARIANTS if column != "query"]
    subset = records_df[records_df["query_clean"].isin(test_queries)]
    if any(column not in subset.columns for column in required_columns):
        return False
    return not (subset[required_columns].isna().any(axis=1) | subset[required_columns].astype(str).apply(lambda col: col.str.strip()).eq("").any(axis=1)).any()

## Run The Experiment

This cell learns subprocess and task weights from the first split for UC1 and UC2. It then averages the learned UC1/UC2 weights by level and applies those learned weights to all UC3 queries without training on UC3. As in the earlier notebook, process-level evaluation reuses subprocess weights because process level has only one query.

For each level, the query set is split into train and test subsets:

$$
Q = Q_{train} \cup Q_{test}, \quad Q_{train} \cap Q_{test} = \emptyset
$$

Weights are learned only from $Q_{train}$:

$$
\mathbf{w}^{*} = \operatorname{LearnWeights}(Q_{train})
$$

The learned weights are evaluated on held-out queries:

$$
\hat{G}(q) = \operatorname{Top}_{k}\left(\operatorname{rank}_{\mathbf{w}^{*}}(q)\right), \quad q \in Q_{test}
$$

where $\hat{G}(q)$ is the predicted relevant set for query $q$.

The main comparison is between:

- `first_pass_judge_variants`: learned linear weights applied to the first-pass judge variants.
- `second_test_judge_variants`: learned linear weights applied after an additional judge refinement on the test split.

For process level, there is usually only one query. Therefore, no separate process-level model is learned. Instead, the subprocess weights are reused:

$$
\mathbf{w}_{process}^{*} = \mathbf{w}_{subprocess}^{*}
$$

In [27]:
def ranked_outputs_to_rows(use_case, level, stage, ranked_outputs, documents):
    rows = []
    for query, scored_docs in ranked_outputs.items():
        for rank, (doc_id, score) in enumerate(scored_docs, start=1):
            rows.append(
                {
                    "use_case": use_case,
                    "level": level,
                    "stage": stage,
                    "query": query,
                    "rank": rank,
                    "doc_id": doc_id,
                    "score": score,
                    "rel_text": clean_text(documents[doc_id].text),
                }
            )
    return rows


def run_level_experiment(use_case, level, records_df, override_weights=None, override_source=None, use_all_queries_as_test=False):
    top_k = LEVEL_TO_TOP_K[level]
    documents = load_corpus(str(corpus_path_for(use_case)))
    bm25_index = build_bm25_index(str(corpus_path_for(use_case)))
    gold_lookup = build_gold_lookup(use_case, level, documents)
    level_records = records_df[records_df["level"] == level].copy()
    queries = [query for query in level_records["query_clean"].tolist() if query in gold_lookup]
    train_queries, test_queries, split_note = split_queries(queries)
    if use_all_queries_as_test:
        train_queries = []
        test_queries = queries
        split_note = "all_queries_external_weights"

    if override_weights is None:
        x_train, y_train = build_examples(
            records_df=level_records,
            bm25_index=bm25_index,
            gold_lookup=gold_lookup,
            query_subset=train_queries,
            top_k=top_k,
            variant_columns=JUDGE_VARIANTS,
        )
        weights, weight_source = learn_weights(x_train, y_train, JUDGE_VARIANTS)
        train_query_count = len(train_queries)
    else:
        weights = np.array(override_weights, dtype=float)
        weight_source = override_source or "reused_weights"
        split_note = f"{split_note}_with_reused_weights"
        train_query_count = 0

    first_predictions, first_ranked_outputs = rank_with_weights(
        records_df=level_records,
        bm25_index=bm25_index,
        query_subset=test_queries,
        top_k=top_k,
        weights=weights,
        variant_columns=JUDGE_VARIANTS,
    )
    first_metrics = evaluate_predictions(first_predictions, gold_lookup)

    metric_rows = [
        {
            "use_case": use_case,
            "level": level,
            "stage": "first_pass_judge_variants",
            "split_note": split_note,
            "weight_source": weight_source,
            "train_queries": train_query_count,
            "test_queries": len(test_queries),
            **first_metrics,
        }
    ]
    prediction_rows = ranked_outputs_to_rows(use_case, level, "first_pass_judge_variants", first_ranked_outputs, documents)

    if RUN_SECOND_TEST_JUDGE:
        sota_retriever = SotaRetriever([document.text for document in documents])
        level_records, second_note = apply_second_test_judge(
            use_case=use_case,
            level=level,
            records_df=level_records,
            test_queries=test_queries,
            weights=weights,
            documents=documents,
            bm25_index=bm25_index,
            sota_retriever=sota_retriever,
            top_k=top_k,
        )
        second_predictions, second_ranked_outputs = rank_with_weights(
            records_df=level_records,
            bm25_index=bm25_index,
            query_subset=test_queries,
            top_k=top_k,
            weights=weights,
            variant_columns=SECOND_JUDGE_VARIANTS,
        )
        second_metrics = evaluate_predictions(second_predictions, gold_lookup)
        metric_rows.append(
            {
                "use_case": use_case,
                "level": level,
                "stage": "second_test_judge_variants",
                "split_note": f"{split_note}_{second_note}",
                "weight_source": weight_source,
                "train_queries": train_query_count,
                "test_queries": len(test_queries),
                **second_metrics,
            }
        )
        prediction_rows.extend(ranked_outputs_to_rows(use_case, level, "second_test_judge_variants", second_ranked_outputs, documents))

    weight_row = {
        "use_case": use_case,
        "level": level,
        "split_note": split_note,
        "weight_source": weight_source,
        "train_queries": train_query_count,
        "test_queries": len(test_queries),
    }
    for (variant_name, _), weight in zip(JUDGE_VARIANTS, weights):
        weight_row[variant_name] = weight

    return weight_row, metric_rows, prediction_rows


def weights_from_row(weight_row):
    return np.array([weight_row[variant_name] for variant_name, _ in JUDGE_VARIANTS], dtype=float)


def average_weights(weight_rows_for_level):
    matrix = np.array([weights_from_row(row) for row in weight_rows_for_level], dtype=float)
    weights = matrix.mean(axis=0)
    return weights / weights.sum()


weight_rows = []
metric_rows = []
prediction_rows = []
trained_weights_by_level = {"subprocess": [], "task": []}

for use_case in TRAIN_USE_CASES:
    records_df = load_records(use_case)
    records_df = ensure_first_pass_judge_records(use_case, records_df)

    subprocess_weight_row, subprocess_metric_rows, subprocess_prediction_rows = run_level_experiment(use_case, "subprocess", records_df)
    weight_rows.append(subprocess_weight_row)
    trained_weights_by_level["subprocess"].append(subprocess_weight_row)
    metric_rows.extend(subprocess_metric_rows)
    prediction_rows.extend(subprocess_prediction_rows)

    task_weight_row, task_metric_rows, task_prediction_rows = run_level_experiment(use_case, "task", records_df)
    weight_rows.append(task_weight_row)
    trained_weights_by_level["task"].append(task_weight_row)
    metric_rows.extend(task_metric_rows)
    prediction_rows.extend(task_prediction_rows)

    subprocess_weights = weights_from_row(subprocess_weight_row)
    process_weight_row, process_metric_rows, process_prediction_rows = run_level_experiment(
        use_case,
        "process",
        records_df,
        override_weights=subprocess_weights,
        override_source="reused_subprocess_judge_weights",
    )
    weight_rows.append(process_weight_row)
    metric_rows.extend(process_metric_rows)
    prediction_rows.extend(process_prediction_rows)

applied_weights = {
    "subprocess": average_weights(trained_weights_by_level["subprocess"]),
    "task": average_weights(trained_weights_by_level["task"]),
}
applied_weights["process"] = applied_weights["subprocess"]

for use_case in APPLY_WEIGHT_USE_CASES:
    records_df = load_records(use_case)
    records_df = ensure_first_pass_judge_records(use_case, records_df)

    for level in ["subprocess", "task", "process"]:
        applied_weight_row, applied_metric_rows, applied_prediction_rows = run_level_experiment(
            use_case,
            level,
            records_df,
            override_weights=applied_weights[level],
            override_source=f"applied_average_{'_'.join(TRAIN_USE_CASES)}_{level}_weights",
            use_all_queries_as_test=True,
        )
        applied_weight_row["trained_on"] = ",".join(TRAIN_USE_CASES)
        weight_rows.append(applied_weight_row)
        metric_rows.extend(applied_metric_rows)
        prediction_rows.extend(applied_prediction_rows)

weights_df = pd.DataFrame(weight_rows)
metrics_df = pd.DataFrame(metric_rows)
predictions_df = pd.DataFrame(prediction_rows)

weights_df

uc1: first-pass judge columns already available.
[uc1 / subprocess] second-pass test judge for query: There are various options for submitting a claim and logging the information int
[uc1 / subprocess] second-pass test judge for query: The claims consultant is reviewing the information received for a travel insuran
[uc1 / subprocess] second-pass test judge for query: Once a claim is approved for compensation, the financial department pays out the
[uc1 / subprocess] second-pass test judge for query: When a claim is closed, the insurer's control unit can still conducts a closed f
[uc1 / task] second-pass test judge for query: When a customer emails the insurance company a paper form of the travel insuranc
[uc1 / task] second-pass test judge for query: When a customer calls the insurance company on the phone to start the travel ins
[uc1 / task] second-pass test judge for query: When the claims consultant from the insurance company receives a claim via call,
[uc1 / task] second-pass test j

,use_case,level,split_note,weight_source,train_queries,test_queries,baseline,judge_legal_terminology_rewrite,judge_regulatory_compliance_query,judge_contract_clause_query,judge_risk_scenario_query,trained_on
0,uc1,subprocess,random_half_split,linear_logistic_coefficients,3,4,0.144029,0.000000,0.395975,0.227592,0.232404,NaN
1,uc1,task,random_half_split,linear_logistic_coefficients,14,15,0.370040,0.087811,0.023283,0.191547,0.327319,NaN
2,uc1,process,single_query_reused_with_reused_weights,reused_subprocess_judge_weights,0,1,0.144029,0.000000,0.395975,0.227592,0.232404,NaN
3,uc2,subprocess,random_half_split,linear_logistic_coefficients,3,4,0.141471,0.214638,0.372786,0.016863,0.254242,NaN
4,uc2,task,random_half_split,linear_logistic_coefficients,9,10,0.264170,0.089698,0.267819,0.057296,0.321016,NaN
5,uc2,process,single_query_reused_with_reused_weights,reused_subprocess_judge_weights,0,1,0.141471,0.214638,0.372786,0.016863,0.254242,NaN
6,uc3,subprocess,all_queries_external_weights_with_reused_weights,applied_average_uc1_uc2_subprocess_weights,0,5,0.142750,0.107319,0.384380,0.122228,0.243323,"uc1,uc2"
7,uc3,task,all_queries_external_weights_with_reused_weights,applied_average_uc1_uc2_task_weights,0,20,0.317105,0.088754,0.145551,0.124422,0.324167,"uc1,uc2"
8,uc3,process,all_queries_external_weights_with_reused_weights,applied_average_uc1_uc2_process_weights,0,1,0.142750,0.107319,0.384380,0.122228,0.243323,"uc1,uc2"


## Held-Out Metrics

This table shows precision, recall, and F1 on the held-out test split. When `RUN_SECOND_TEST_JUDGE` is enabled, each level gets an additional row for the second test-set judge pass.

For each held-out query $q$, the notebook compares the predicted set $\hat{G}(q)$ with the gold-standard relevant set $G(q)$:

$$
TP(q) = |\hat{G}(q) \cap G(q)|
$$

$$
FP(q) = |\hat{G}(q) \setminus G(q)|
$$

$$
FN(q) = |G(q) \setminus \hat{G}(q)|
$$

The totals are summed over all held-out queries:

$$
TP = \sum_{q \in Q_{test}} TP(q), \quad FP = \sum_{q \in Q_{test}} FP(q), \quad FN = \sum_{q \in Q_{test}} FN(q)
$$

Precision measures how many retrieved documents were relevant:

$$
\operatorname{Precision} = \frac{TP}{TP + FP}
$$

Recall measures how many gold relevant documents were recovered:

$$
\operatorname{Recall} = \frac{TP}{TP + FN}
$$

F1 is the harmonic mean of precision and recall:

$$
F_1 = \frac{2 \cdot \operatorname{Precision} \cdot \operatorname{Recall}}{\operatorname{Precision} + \operatorname{Recall}}
$$

Because the goal of this experiment is very high recall, the most important columns are `true_positives`, `false_negatives`, and `recall`.

In [28]:
display_columns = [
    "use_case",
    "level",
    "stage",
    "weight_source",
    "train_queries",
    "test_queries",
    "true_positives",
    "false_positives",
    "false_negatives",
    "precision",
    "recall",
    "f1",
]

metrics_display = metrics_df[display_columns].copy()
metrics_display[["precision", "recall", "f1"]] = metrics_display[["precision", "recall", "f1"]].round(3)
metrics_display

,use_case,level,stage,weight_source,train_queries,test_queries,true_positives,false_positives,false_negatives,precision,recall,f1
0,uc1,subprocess,first_pass_judge_variants,linear_logistic_coefficients,3,4,9,111,29,0.075,0.237,0.114
1,uc1,subprocess,second_test_judge_variants,linear_logistic_coefficients,3,4,5,115,33,0.042,0.132,0.063
2,uc1,task,first_pass_judge_variants,linear_logistic_coefficients,14,15,13,212,60,0.058,0.178,0.087
3,uc1,task,second_test_judge_variants,linear_logistic_coefficients,14,15,13,212,60,0.058,0.178,0.087
4,uc1,process,first_pass_judge_variants,reused_subprocess_judge_weights,0,1,23,77,26,0.230,0.469,0.309
5,uc1,process,second_test_judge_variants,reused_subprocess_judge_weights,0,1,15,85,34,0.150,0.306,0.201
6,uc2,subprocess,first_pass_judge_variants,linear_logistic_coefficients,3,4,20,100,20,0.167,0.500,0.250
7,uc2,subprocess,second_test_judge_variants,linear_logistic_coefficients,3,4,18,102,22,0.150,0.450,0.225
8,uc2,task,first_pass_judge_variants,linear_logistic_coefficients,9,10,22,128,43,0.147,0.338,0.205
9,uc2,task,second_test_judge_variants,linear_logistic_coefficients,9,10,21,129,44,0.140,0.323,0.195


## Export Results

The export contains learned weights, held-out metrics, and ranked predictions for both available stages.

The exported sheets correspond to the main mathematical objects in the notebook:

- `learned_weights` stores $\mathbf{w}^{*}$ for each use case and level.
- `heldout_metrics` stores $TP$, $FP$, $FN$, precision, recall, and $F_1$.
- `ranked_predictions` stores the ranked predictions induced by $s_{\mathbf{w}}(q,d)$ or $s_{\mathbf{w}}^{(2)}(q,d)$.

In [29]:
export_path = project_root / "linear_rrf_weight_model_llm_judge_results.xlsx"

with pd.ExcelWriter(export_path) as writer:
    weights_df.to_excel(writer, sheet_name="learned_weights", index=False)
    metrics_df.to_excel(writer, sheet_name="heldout_metrics", index=False)
    predictions_df.to_excel(writer, sheet_name="ranked_predictions", index=False)

print(f"Exported {export_path}")

Exported /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval/linear_rrf_weight_model_llm_judge_results.xlsx
